In [1]:
!pip install -U sentence-transformers pinecone-client openai pymupdf torch transformers accelerate bitsandbytes flask flask-cors python-dotenv

In [2]:
pip install --upgrade sympy transformers langchain


  Using cached langchain-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_core-1.2.7-py3-none-any.whl.metadata (3.7 kB)
  Using cached langsmith-0.6.4-py3-none-any.whl.metadata (15 kB)
Using cached langchain-1.2.6-py3-none-any.whl (108 kB)
Using cached langchain_core-1.2.7-py3-none-any.whl (490 kB)
Using cached langsmith-0.6.4-py3-none-any.whl (283 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.0.87

    Uninstalling langsmith-0.0.87:

      Successfully uninstalled langsmith-0.0.87

   ---------------------------------------- 0/3 [langsmith]
   ---------------------------------------- 0/3 [langsmith]
   ---------------------------------------- 0/3 [langsmith]
   ---------------------------------------- 0/3 [langsmith]
   ---------------------------------------- 0/3 [langsmith]
  Attempting uninstall: langchain-core
   ---------------------------------------- 0/3 [langsmith]
    Found existing installation: langchain-core 0.1.23
  

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.20 requires langchain-core<0.2,>=0.1.21, but you have langchain-core 1.2.7 which is incompatible.
langchain-community 0.0.20 requires langsmith<0.1,>=0.0.83, but you have langsmith 0.6.4 which is incompatible.


In [3]:
pip install langchain==0.0.352 langchain-community==0.0.20


  Using cached langchain-0.0.352-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
  Using cached langsmith-0.0.92-py3-none-any.whl.metadata (9.9 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.1.52-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_core-0.1.51-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_core-0.1.50-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_core-0.1.49-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_core-0.1.48-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_core-0.1.47-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_core-0.1.46-py3-none-any.whl.metadata (5.9 kB)
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This co

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.1 requires langchain-core<2.0.0,>=1.2.5, but you have langchain-core 0.1.23 which is incompatible.
langchain-classic 1.0.1 requires langsmith<1.0.0,>=0.1.17, but you have langsmith 0.0.87 which is incompatible.
langchain-text-splitters 1.1.0 requires langchain-core<2.0.0,>=1.2.0, but you have langchain-core 0.1.23 which is incompatible.
langgraph-checkpoint 4.0.0 requires langchain-core>=0.2.38, but you have langchain-core 0.1.23 which is incompatible.
langgraph-prebuilt 1.0.6 requires langchain-core>=1.0.0, but you have langchain-core 0.1.23 which is incompatible.


In [4]:
from docx import Document
import pinecone
import torch
import openai
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
import fitz
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from flask import Flask, request, jsonify
from flask_cors import CORS
import unicodedata
import pickle
from dotenv import load_dotenv
import os

c:\Users\ehabq\Documents\GitHub\Warasat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
load_dotenv("keys.env") 
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [6]:
file_path = "A Practical Guide to Islamic Law of Inheritance_250313_011139.pdf"

with fitz.open(file_path) as doc:
    text_data = []
    for page in doc:
        text_data.append(page.get_text("text"))  # Extract raw text

full_text = "\n".join(text_data)

print(full_text[:1000])  # Print the first 1000 characters


1 
 
A Practical Guide to  
Islamic Laws of Inheritance 
 
 
By Muhammad Ahsan Zafar 
Dedicated to my teacher: Eng. Malik Bashir Ahmad Baghvi 
 
 
 
What’s inside? 
1. Terminologies and abbreviations 
2. Steps of wealth distribution 
3. Calculating share of heirs 
4. Sample cases 
5. Exceptional cases - Problem of excess & deficiency 
6. Sources of knowledge for calculations – References 
7. Making a Will 
8. Practice questions 
Please note; Material in this document is for teaching and learning purpose and should not be edited 
without permission. Contact: AHSANZZ@GMAIL.COM 

2 
 
Notes 
 
 
 
 
 
 
 
 
 
 
Problem solving template 
 
Name:                                                  Date of death:                                   Distributable Wealth:  

3 
 
TERMINOLOGIES 
Mirath - Gross Inheritance: All movable or immovable property left behind by deceased whether the 
deceased earned it, inherited it or was gifted this property.    
Warith - Heir: A relative who may potentia

In [7]:

full_text = unicodedata.normalize("NFKD", full_text)

full_text = full_text.encode("ascii", "ignore").decode()

print(full_text[:1000])

1 
 
A Practical Guide to  
Islamic Laws of Inheritance 
 
 
By Muhammad Ahsan Zafar 
Dedicated to my teacher: Eng. Malik Bashir Ahmad Baghvi 
 
 
 
Whats inside? 
1. Terminologies and abbreviations 
2. Steps of wealth distribution 
3. Calculating share of heirs 
4. Sample cases 
5. Exceptional cases - Problem of excess & deficiency 
6. Sources of knowledge for calculations  References 
7. Making a Will 
8. Practice questions 
Please note; Material in this document is for teaching and learning purpose and should not be edited 
without permission. Contact: AHSANZZ@GMAIL.COM 

2 
 
Notes 
 
 
 
 
 
 
 
 
 
 
Problem solving template 
 
Name:                                                  Date of death:                                   Distributable Wealth:  

3 
 
TERMINOLOGIES 
Mirath - Gross Inheritance: All movable or immovable property left behind by deceased whether the 
deceased earned it, inherited it or was gifted this property.    
Warith - Heir: A relative who may potentiall

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
chunks = splitter.split_text(full_text)

embeddings = model.encode(chunks, convert_to_tensor=True)

with open("embeddings.pkl", "wb") as f:
    pickle.dump((chunks, embeddings), f)

print(f"Fixed Embedding Shape: {embeddings.shape}")


Fixed Embedding Shape: torch.Size([46, 384])


In [9]:
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY)

index_name = "inheritance-rules-index"

existing_indexes = [index.name for index in pc.list_indexes()]

if index_name in existing_indexes:
    print(f"Index '{index_name}' already exists.")
else:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=pinecone.ServerlessSpec(cloud="aws", region="us-east-1")
    )

In [10]:
index = pc.Index(index_name)

vectors = []
for i, chunk in enumerate(chunks):
    embedding = model.encode(chunk, convert_to_tensor=False).tolist()
    vectors.append((str(i), embedding, {"text": chunk}))

batch_size = 100
for i in range(0, len(vectors), batch_size):
    index.upsert(vectors[i : i + batch_size])

print(f"Stored {len(vectors)} text chunks in Pinecone!")


Stored 46 text chunks in Pinecone!


In [11]:
def query_pinecone(question, top_k=3, threshold=0.3):
    query_embedding = model.encode(question, convert_to_numpy=True).astype(np.float32).tolist()

    results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True, include_values=False)

    filtered_responses = [
        match["metadata"]["text"] for match in results["matches"] if match["score"] >= threshold
    ]

    return filtered_responses if filtered_responses else ["Unable to answer this question"]

user_question = "Explain Masla Aul?"
retrieved_texts = query_pinecone(user_question)

for i, text in enumerate(retrieved_texts):
    print(f"Answer {i+1}:\n{text}\n")


Answer 1:
11 
 
2. Problem of Deficiency (Masla-Aul): Another situation may arise where the assigned fixed shares 
of heirs may exceed the denominator value. This case is rare. When this happens, the denominator is 
increased to the sum of all the shares, hence all heirs have a reduction in their share proportional to 
their share ratio. Having spouse heirs does not change the calculation method, in contrast to Masla-
Radd. 
Example: Zainab died leaving behind a husband, 2 daughters, father, mother and 1 real brother. Her 
distributable wealth is $ 50,000 
 
INELIGIBLITY FOR INHERITANCE 
1. HOMICIDE: The murderer of the deceased will be disqualified from his/ her inheritance, even if he/

Answer 2:
male relatives related through a chain of males (exception: real sister and paternal sister)     
Dhil-irham (DI)  3rd tier of heirs after Dhil-Furooz and Asbah. If there is still left over property after 
distribution to Dhil-Furooz, and there are no Asbah then Dhil-irham may be entitled to

In [12]:
app = Flask(__name__)
CORS(app)  

client = openai.OpenAI(api_key=OPENAI_API_KEY)

def generate_answer(query, context):
    
    prompt = f"Use the following information to answer the query:\n\n{context}\n\nQuery: {query}\nAnswer:"

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an AI assistant that provides Islamic inheritance answers based on provided documents."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=100
    )

    return response.choices[0].message.content  

@app.route("/get_answer", methods=["POST"])
def get_answer():
    """
    API endpoint that takes a user query, retrieves context, and generates a response.
    """
    try:
        data = request.json
        query = data.get("query", "")

        if not query:
            return jsonify({"error": "Query is required"}), 400

        retrieved_context = query_pinecone(query, top_k=3)

        final_answer = generate_answer(query, retrieved_context)

        return jsonify({"answer": final_answer})  

    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000)



 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.116:5000
Press CTRL+C to quit
